<a href="https://colab.research.google.com/github/niteshsahdeveloper/gen-ai-bootcamp/blob/master/Day_3_homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers faiss-cpu ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 93.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
import ipywidgets as widgets
from IPython.display import display, HTML

# -------------------------------------------------------------------------
# 1. Load the LLM Embedding Model (all-MiniLM-L6-v2)
# -------------------------------------------------------------------------
print("Loading embedding model (all-MiniLM-L6-v2)...")
model = SentenceTransformer("all-MiniLM-L6-v2")

# -------------------------------------------------------------------------
# 2. Select Use Case: Use Case 7 (IT Support Tickets)
# Create the records/chunks to be stored in the vector database
# -------------------------------------------------------------------------
tickets = [
    {
        "ticket_id": "IT-101",
        "category": "Network / VPN",
        "issue": "VPN connection drops every 10 minutes when working remotely",
        "resolution": "Update Cisco AnyConnect to latest patch, flush DNS cache, and set network MTU size to 1350."
    },
    {
        "ticket_id": "IT-102",
        "category": "Email / Authentication",
        "issue": "Outlook prompting for password repeatedly despite correct credentials",
        "resolution": "Clear Microsoft credentials from Windows Credential Manager and enable modern authentication in registry."
    },
    {
        "ticket_id": "IT-103",
        "category": "Hardware / Performance",
        "issue": "Laptop battery drains rapidly and overheats while running containers",
        "resolution": "Cap Docker desktop resource allocations to 4 CPU cores, 8GB RAM, and update BIOS thermal tables."
    },
    {
        "ticket_id": "IT-104",
        "category": "DevOps / Git",
        "issue": "Cannot push commits to GitHub repository: Permission denied (publickey)",
        "resolution": "Generate a new ED25519 SSH key, launch ssh-agent, add identity via ssh-add, and register key in GitHub settings."
    },
    {
        "ticket_id": "IT-105",
        "category": "OS / Crash",
        "issue": "Blue Screen of Death (BSOD) showing error DRIVER_IRQL_NOT_LESS_OR_EQUAL",
        "resolution": "Roll back recent Wi-Fi and display drivers in Device Manager, then run Windows Memory Diagnostic."
    },
    {
        "ticket_id": "IT-106",
        "category": "Collaboration / Video",
        "issue": "Screen sharing shows a black screen to meeting participants on macOS",
        "resolution": "Go to System Settings > Privacy & Security > Screen Recording, uncheck the app, and re-enable permissions."
    },
    {
        "ticket_id": "IT-107",
        "category": "Peripherals",
        "issue": "Network printer shows offline status and print spooler is stuck",
        "resolution": "Restart the Print Spooler service via services.msc and remove stuck print jobs in spool/PRINTERS directory."
    }
]

# Convert records into structured text chunks for embedding
chunks = [f"Issue: {t['issue']} | Category: {t['category']} | Resolution: {t['resolution']}" for t in tickets]

# -------------------------------------------------------------------------
# 3. Convert chunks into vectors & store them in FAISS
# -------------------------------------------------------------------------
print("Encoding chunks into vector space...")
ticket_vectors = model.encode(chunks, convert_to_numpy=True)

# Normalize vectors to unit length so that Inner Product (IP) equals Cosine Similarity
faiss.normalize_L2(ticket_vectors)

# Create FAISS IndexFlatIP (Inner Product)
dim = ticket_vectors.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(ticket_vectors)

print(f"FAISS index built successfully with {index.ntotal} vectors of dimension {dim}!\n")

# -------------------------------------------------------------------------
# 4 & 5 & 6. Query Search Function (k-NN Cosine Similarity + % Similarity)
# -------------------------------------------------------------------------
def search_similar_tickets(user_query: str, k: int = 3):
    # Vectorize query and normalize for cosine similarity
    query_vector = model.encode([user_query], convert_to_numpy=True)
    faiss.normalize_L2(query_vector)

    # Search the top-k nearest neighbors
    cos_similarities, indices = index.search(query_vector, k)

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], cos_similarities[0]), start=1):
        matched = tickets[idx]
        # Cosine similarity range [-1, 1] scaled to percentage [0%, 100%]
        similarity_pct = max(0.0, float(score)) * 100

        results.append({
            "Rank": rank,
            "Ticket ID": matched["ticket_id"],
            "Category": matched["category"],
            "Similarity (%)": f"{similarity_pct:.2f}%",
            "Raw Cosine Score": f"{float(score):.4f}",
            "Matched Issue": matched["issue"],
            "Recommended Resolution": matched["resolution"]
        })

    return pd.DataFrame(results)

# -------------------------------------------------------------------------
# Interactive UI using Colab Widgets
# -------------------------------------------------------------------------
display(HTML("<h3>🛠️ IT Support Ticket - FAISS Vector Search Demo</h3>"))

query_input = widgets.Textarea(
    value="My remote VPN client keeps disconnecting while I am trying to work",
    description="Query:",
    layout=widgets.Layout(width="80%", height="80px")
)

k_slider = widgets.IntSlider(
    value=3,
    min=1,
    max=len(tickets),
    step=1,
    description="Top-k:",
    continuous_update=False
)

search_button = widgets.Button(
    description="Search Tickets",
    button_style="primary",
    icon="search"
)

output_area = widgets.Output()

def on_search_clicked(b):
    with output_area:
        output_area.clear_output()
        df = search_similar_tickets(query_input.value, k=k_slider.value)
        display(HTML("<b>Top Matching Tickets Found:</b>"))
        display(df)

search_button.on_click(on_search_clicked)

display(query_input, k_slider, search_button, output_area)

# Trigger once on initial run
on_search_clicked(None)

Loading embedding model (all-MiniLM-L6-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Encoding chunks into vector space...
FAISS index built successfully with 7 vectors of dimension 384!



Textarea(value='My remote VPN client keeps disconnecting while I am trying to work', description='Query:', lay…

IntSlider(value=3, continuous_update=False, description='Top-k:', max=7, min=1)

Button(button_style='primary', description='Search Tickets', icon='search', style=ButtonStyle())

Output()